In [ ]:
# DADDI ADDOUN Sami
# Matrix number: 24223007
# CODE 2: VLM

import numpy as np
import matplotlib.pyplot as plt
import os
import json
import cv2
import sys
import pickle
import time
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import seaborn as sns
!pip install gensim
from gensim.models import KeyedVectors

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, LSTM, Embedding, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- CONFIGURATION ---
BASE_PATH = 'YOUR_PATH_TO_DATA'
IMAGE_PATH = os.path.join(BASE_PATH, 'VQA_RAD_Image_Folder')
JSON_PATH = os.path.join(BASE_PATH, 'VQA_RAD_Dataset_Public.json')
W2V_PATH = 'PATH_TO_W2V_File'
RESULTS_PATH = 'YOUR_PATH_TO_RESULTS'

IMG_SIZE = (224, 224)
NUM_ANS_TOP = 15
MAX_Q_LEN = 20
EMBED_DIM = 300
N_FOLDS = 5

# ---  DATA LOADING ---
print("--- Loading VLM Data ---")
with open(JSON_PATH, 'r') as f: raw_data = json.load(f)

# Top 15 Logic
counts = {}
for item in raw_data:
    ans = str(item['answer']).lower()
    counts[ans] = counts.get(ans, 0) + 1
top_ans = [x[0] for x in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:NUM_ANS_TOP]]
atoi = {w: i for i, w in enumerate(top_ans)}

# Load Data
X_images, Y_ids, questions_text = [], [], []
filenames = []

print("Preparing Images & Texts...")
for item in tqdm(raw_data):
    #normalize answer to lowercase
    ans = str(item['answer']).lower()
    #we only process samples where the answer is in our top frequent vocabulary
    if ans in atoi:
        path = os.path.join(IMAGE_PATH, item['image_name'])
        img = cv2.imread(path)
        if img is not None:

            #resizing the image to fit ResNet50 input shape
            img = cv2.resize(img, IMG_SIZE)

            # colorspace conversion from BGR to RGB (TensorFlow/Matplotlib standard)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            #append the processed data to our lists
            X_images.append(img)
            Y_ids.append(atoi[ans])

            #store metadata
            questions_text.append(str(item['question']).lower())
            filenames.append(item['image_name']) # Keep track

X_images = preprocess_input(np.array(X_images))
Y_ids = np.array(Y_ids)
Y_one_hot = to_categorical(Y_ids, num_classes=NUM_ANS_TOP)
filenames = np.array(filenames)

# ---  NLP PREP ---
print("Tokenization...")
# convert raw question strings into a list of individuals tokens (words)
tokens = [word_tokenize(q) for q in questions_text]

# build vocabulary dictionnary, where each unique word has a unique integer index
word_index = {}
idx = 1 #idx 0 is reserved to padding
for seq in tokens:
    for w in seq:
        if w not in word_index: word_index[w] = idx; idx += 1
#pad sequences to ensure fixed sequences length
X_text = pad_sequences([[word_index[w] for w in seq] for seq in tokens],
                        maxlen=MAX_Q_LEN, padding='post')
#prepare embedding matrix
vocab_size = len(word_index) + 1
embed_matrix = np.zeros((vocab_size, EMBED_DIM))

# Loading pre-trained Word2Vec embeddings and injecting its vector into our matrix
if os.path.exists(W2V_PATH):
    print("Loading Word2Vec...")
    w2v = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True)
    for w, i in word_index.items():
        if w in w2v: embed_matrix[i] = w2v[w]
else:
    print("warning: using random embeddings")

# ---  MODEL ---
def build_vlm_model():

    # ResNet50, all layers frozen, without the classification layer
    in_img = Input((224,224,3))
    base = ResNet50(weights='imagenet', include_top=False, input_tensor=in_img)
    for l in base.layers: l.trainable = False

    #convert 3D feature map into 1D vector
    x_img = GlobalAveragePooling2D()(base.output)

    #projecting feature to a lower dimension
    x_img = Dense(512, activation='relu')(x_img)
    x_img = Dropout(0.5)(x_img)

    #input layer for tokenized question sequences
    in_txt = Input((MAX_Q_LEN,))

    # embedding layer pre-trained with Word2Vec weights ( we keep embedding static)
    emb = Embedding(vocab_size, EMBED_DIM, weights=[embed_matrix],
                    trainable=False, mask_zero=True)(in_txt)
    x_txt = LSTM(256)(emb)
    x_txt = Dropout(0.5)(x_txt)

    # multimodal fusion
    # we project both modalities to the same dimension then we use element wise multiplication to fuse them
    fusion = Multiply()([Dense(256, activation='relu')(x_img),
                         Dense(256, activation='relu')(x_txt)])
    # Post-fusion processing
    x = Dense(256, activation='relu')(fusion)
    x = Dropout(0.5)(x)

    # Output layer: Softmax activation for multi-class classification
    out = Dense(NUM_ANS_TOP, activation='softmax')(x)
    model = Model([in_img, in_txt], out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ---  CROSS-VALIDATION ---
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
all_true, all_pred = [], []
all_filenames, all_questions = [], []
inference_times = []
histories = []

print(f"\n--- Starting VLM CV ---")

fold = 1
for train_idx, val_idx in kfold.split(X_images, Y_ids):
    print(f"\nFold {fold}/{N_FOLDS}...")
    K.clear_session()
    # data splitting
    Xi_tr, Xi_val = X_images[train_idx], X_images[val_idx]
    Xt_tr, Xt_val = X_text[train_idx], X_text[val_idx]
    Y_tr, Y_val = Y_one_hot[train_idx], Y_one_hot[val_idx]

    # Metadata
    val_fnames = filenames[val_idx]
    val_qs = np.array(questions_text)[val_idx]

    model = build_vlm_model()
    es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    #training the model
    h = model.fit([Xi_tr, Xt_tr], Y_tr, validation_data=([Xi_val, Xt_val], Y_val),
              epochs=30, batch_size=32, callbacks=[es], verbose=1)
    histories.append(h.history)

    # --- INFERENCE TIME ---
    # Warm-up step to initialize GPU (avoids cold start bias)
    _ = model.predict([Xi_val[:1], Xt_val[:1]], verbose=0)

    # inference measurment
    start_time = time.time()
    preds_probs = model.predict([Xi_val, Xt_val], verbose=0)
    end_time = time.time()

    avg_time = ((end_time - start_time) / len(Xi_val)) * 1000
    inference_times.append(avg_time)

    preds = np.argmax(preds_probs, axis=1)
    trues = np.argmax(Y_val, axis=1)

    all_pred.extend(preds)
    all_true.extend(trues)
    all_filenames.extend(val_fnames)
    all_questions.extend(val_qs)

    fold += 1

# ---  SAVING ---
model_params = model.count_params()
global_acc = accuracy_score(all_true, all_pred)
macro_f1 = f1_score(all_true, all_pred, average='macro')
mean_inference_time = np.mean(inference_times)

is_closed = np.array([top_ans[i] in ['yes', 'no'] for i in all_true])
acc_closed = accuracy_score(np.array(all_true)[is_closed], np.array(all_pred)[is_closed])
acc_open = accuracy_score(np.array(all_true)[~is_closed], np.array(all_pred)[~is_closed])

results_data = {
    'model': 'VLM_SOTA',
    'accuracy': global_acc,
    'macro_f1': macro_f1,
    'acc_closed': acc_closed,
    'acc_open': acc_open,
    'all_true': all_true,
    'all_pred': all_pred,
    'classes': top_ans,
    'filenames': all_filenames,
    'questions': all_questions,
    'history': histories,
    'inference_time_ms': mean_inference_time,
    'params': model_params
}

save_file = os.path.join(RESULTS_PATH, 'vlm_results.pkl')
with open(save_file, 'wb') as f:
    pickle.dump(results_data, f)

print(f"\n VLM Results saved to: {save_file}")
print(f"\n Average Inference Time: {mean_inference_time:.2f} ms/sample")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.6 MB/s eta 0:00:00
--- Loading VLM Data ---
Preparing Images & Texts...


100%|██████████| 2248/2248 [02:01<00:00, 18.53it/s] 


Tokenization...
Loading Word2Vec...

--- Starting VLM CV ---

Fold 1/5...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 23s 286ms/step - accuracy: 0.3835 - loss: 1.9006 - val_accuracy: 0.4964 - val_loss: 1.1290
Epoch 2/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 161ms/step - accuracy: 0.5247 - loss: 1.1772 - val_accuracy: 0.5217 - val_loss: 1.0089
Epoch 3/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 165ms/step - accuracy: 0.5722 - loss: 0.9911 - val_accuracy: 0.5399 - val_loss: 1.0071
Epoch 4/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 162ms/step - accuracy: 0.5600 - loss: 0.9706 - val_accuracy: 0.5435 - val_loss: 0.9315
Epoch 5/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 164ms/step - accuracy: 0.6946 - loss: 0.7567 - val_accuracy: 0.5833 - val_loss: 0.8227
Epoch 6/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 168ms/step - accuracy: 0.6904 - loss: 0.7075 - val_accuracy: 0.6087 - val_loss: 0.8179
Epoch 7/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 164ms/step - accuracy: 0.7422 - loss: 0.6182 - val_accuracy: 0.6